# MBG YouTube Sentiment Analysis & Text Mining

Analisis sentimen dan tren diskusi pada komentar YouTube terkait **Makan Bergizi Gratis (MBG)** menggunakan **Natural Language Processing (NLP)** dan **Deep Learning**.

> **Data source:** komentar YouTube yang dikumpulkan secara programatis.
>
> **Important:** dataset yang digunakan notebook ini adalah milik project ini sendiri. Hasil analisis harus dihitung ulang dari dataset ini dan tidak menggunakan hasil dari notebook/project lain.

## 1. Project Objective

Project ini bertujuan untuk:

1. Menganalisis karakteristik dan pola diskusi pada komentar YouTube terkait MBG.
2. Melakukan text preprocessing untuk menyiapkan komentar bahasa Indonesia.
3. Menganalisis distribusi sentimen **positive**, **negative**, dan **neutral** bila label tersedia.
4. Membangun model klasifikasi **positive vs negative** menggunakan Bidirectional RNN.
5. Mengevaluasi model menggunakan accuracy, precision, recall, F1-score, dan confusion matrix.
6. Mengidentifikasi perubahan volume diskusi dari waktu ke waktu.

**Catatan metodologi:** sampel komentar YouTube tidak otomatis mewakili opini seluruh masyarakat Indonesia.

## 2. Data Preparation

Notebook membaca file:

`data/mbg_comments_labeled.csv`

Struktur dataset yang digunakan:
- `row_id` — ID baris pada dataset labeling
- `comment` — komentar YouTube asli
- `comment_clean` — komentar setelah cleaning awal
- `video_title` — judul video sumber komentar
- `channel_title` — nama channel sumber komentar
- `published_at` — waktu publikasi komentar
- `sentiment` — label sentimen

Notebook menggunakan dataset YouTube yang sudah dikumpulkan dan diproses sebelumnya. Notebook tidak melakukan scraping ulang dan tidak menggunakan Tweepy/Twitter-X.

In [ ]:
# Environment setup
# Install NLP dependencies only when they are missing from the active notebook kernel.
import sys
import subprocess
import importlib.util

packages = [("Sastrawi", "Sastrawi"), ("nltk", "nltk")]
for package_name, module_name in packages:
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", package_name
        ])

import nltk
nltk.download("stopwords", quiet=True)
print("Environment setup complete.")


In [ ]:
from pathlib import Path
import re
import html
import json
import random
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_PATH = Path("data/mbg_comments_labeled.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset tidak ditemukan di {DATA_PATH.resolve()}. "
        "Pastikan notebook dijalankan dari root repository."
    )

YT_comments = pd.read_csv(DATA_PATH)

print(f"Dataset loaded: {YT_comments.shape[0]:,} rows x {YT_comments.shape[1]} columns")
print("Columns:")
print(YT_comments.columns.tolist())

## 3. Data Validation

Sebelum preprocessing dan modeling, notebook memeriksa:
- jumlah baris dan kolom
- missing value
- duplicate berdasarkan `row_id`
- duplicate exact text sebagai informasi tambahan, bukan otomatis dihapus
- validitas tanggal
- distribusi label sentimen

Duplicate teks tidak otomatis dihapus karena komentar yang sama dapat muncul pada lebih dari satu baris. `row_id` digunakan sebagai identifier baris pada dataset labeling.

In [ ]:
required_columns = [
    "row_id",
    "comment",
    "comment_clean",
    "published_at",
    "video_title",
    "channel_title",
    "sentiment"
]

missing_required = [c for c in required_columns if c not in YT_comments.columns]
if missing_required:
    raise ValueError(f"Kolom wajib tidak ditemukan: {missing_required}")

print("Shape:", YT_comments.shape)
print("\nColumns:")
print(YT_comments.columns.tolist())

print("\nMissing values:")
display(YT_comments.isna().sum().sort_values(ascending=False).to_frame("missing"))

if "row_id" in YT_comments.columns:
    print("\nDuplicate row_id:", YT_comments["row_id"].duplicated().sum())

if "comment" in YT_comments.columns:
    print("Duplicate exact comment text:", YT_comments["comment"].duplicated().sum())

YT_comments["published_at"] = pd.to_datetime(
    YT_comments["published_at"], errors="coerce", utc=True
)

print("\nInvalid published_at:", YT_comments["published_at"].isna().sum())

print("\nSentiment values before normalization:")
print(YT_comments["sentiment"].value_counts(dropna=False))

## 4. Normalize Sentiment Labels

Label yang diterima notebook:
- `positive`
- `negative`
- `neutral`

Untuk modeling deep learning, notebook menggunakan **binary classification**:
- `positive = 0`
- `negative = 1`

Label `neutral` tetap dipertahankan pada dataset utama dan hanya dikeluarkan dari dataset binary.

**Catatan labeling:** pada versi dataset saat ini, 1.500 komentar pada `mbg_comments_labeled.csv` telah diberi label dengan pendekatan **AI-assisted semantic labeling**. Label tersebut digunakan sebagai label supervised pada notebook dan bukan hasil anotasi manual oleh human annotator.

In [ ]:
YT_comments["sentiment"] = (
    YT_comments["sentiment"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace({
        "pos": "positive",
        "neg": "negative",
        "netral": "neutral"
    })
)

VALID_LABELS = {"positive", "negative", "neutral"}

invalid_labels = sorted(
    set(YT_comments["sentiment"].dropna().unique()) - VALID_LABELS
)
if invalid_labels:
    print("Warning - unexpected labels:", invalid_labels)

labeled_mask = YT_comments["sentiment"].isin(VALID_LABELS)
YT_labeled = YT_comments.loc[labeled_mask].copy()

LABELS_READY = len(YT_labeled) > 0

print(f"Rows with valid labels: {len(YT_labeled):,} / {len(YT_comments):,}")
print("\nLabel distribution:")
if LABELS_READY:
    print(YT_labeled["sentiment"].value_counts())
else:
    print("Belum ada label valid. Silakan isi kolom 'sentiment' terlebih dahulu.")

BINARY_LABELS_READY = (
    LABELS_READY
    and YT_labeled["sentiment"].isin(["positive", "negative"]).sum() > 0
    and YT_labeled.loc[
        YT_labeled["sentiment"].isin(["positive", "negative"]), "sentiment"
    ].nunique() == 2
)

print("\nBinary positive/negative ready:", BINARY_LABELS_READY)

## 5. Text Preprocessing

Dataset sudah memiliki `comment_clean`, sehingga notebook menggunakannya sebagai basis preprocessing dan tidak menghapus informasi dari kolom metadata.

Tahapan NLP:
1. HTML/entity normalization
2. URL / mention cleanup
3. hashtag symbol removal tetapi kata hashtag dipertahankan
4. case folding
5. punctuation normalization
6. slang normalization
7. stopword removal dengan **negation words dipertahankan**
8. stemming Bahasa Indonesia menggunakan Sastrawi
9. hasil akhir disimpan di `text_preprocessed`

Kata negasi seperti `tidak`, `bukan`, `jangan`, `belum`, dan `gak` sengaja tidak dibuang karena dapat mengubah makna sentimen.

In [ ]:
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

try:
    INDONESIAN_STOPWORDS = set(stopwords.words("indonesian"))
except LookupError:
    import nltk
    nltk.download("stopwords")
    INDONESIAN_STOPWORDS = set(stopwords.words("indonesian"))

NEGATION_WORDS = {
    "tidak", "tak", "bukan", "jangan", "belum",
    "gak", "nggak", "ga", "ngga", "tdk", "kurang"
}
INDONESIAN_STOPWORDS = INDONESIAN_STOPWORDS - NEGATION_WORDS

slang_words = {
    "gk": "gak",
    "ga": "gak",
    "nggak": "gak",
    "ngga": "gak",
    "tdk": "tidak",
    "bkn": "bukan",
    "jgn": "jangan",
    "krn": "karena",
    "knp": "kenapa",
    "knpa": "kenapa",
    "ap": "apa",
    "tp": "tapi",
    "tpi": "tapi",
    "bgt": "banget",
    "bgd": "banget",
    "gt": "gitu",
    "jg": "juga",
    "skg": "sekarang",
    "udh": "sudah",
    "sdh": "sudah",
    "dah": "sudah",
    "yg": "yang",
    "dgn": "dengan",
    "lg": "lagi",
    "tau": "tahu",
    "smua": "semua",
    "org": "orang",
    "dpt": "dapat",
    "msh": "masih",
    "utk": "untuk",
    "lbh": "lebih",
    "dri": "dari",
    "dr": "dari",
    "dgn": "dengan",
    "kyk": "kayak",
    "mkn": "makan",
    "mslh": "masalah",
    "bnyk": "banyak",
    "mlh": "malah",
    "nnti": "nanti",
    "tuh": "itu",
    "cuma": "hanya",
    "abis": "habis",
    "bener": "benar",
    "emg": "memang",
    "emang": "memang"
}

stemmer = StemmerFactory().create_stemmer()

def clean_for_nlp(text):
    text = "" if pd.isna(text) else str(text)
    text = html.unescape(text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"@[A-Za-z0-9_]+", " ", text)
    text = re.sub(r"#", "", text)
    text = text.replace("\n", " ")
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = text.split()
    tokens = [slang_words.get(tok, tok) for tok in tokens]
    tokens = [
        tok for tok in tokens
        if tok not in INDONESIAN_STOPWORDS
        and len(tok) > 1
    ]

    if not tokens:
        return ""

    stemmed = stemmer.stem(" ".join(tokens))
    return re.sub(r"\s+", " ", stemmed).strip()

source_text_col = "comment_clean" if "comment_clean" in YT_comments.columns else "comment"

YT_comments["text_source"] = YT_comments[source_text_col].fillna(YT_comments["comment"])
YT_comments["text_preprocessed"] = YT_comments["text_source"].apply(clean_for_nlp)

YT_comments[["comment", "text_source", "text_preprocessed"]].head(10)

In [ ]:
empty_after_preprocessing = (YT_comments["text_preprocessed"].str.len() == 0).sum()

print(f"Empty text after preprocessing: {empty_after_preprocessing:,}")
print("\nSample preprocessing:")
display(
    YT_comments[["comment", "text_preprocessed"]]
    .sample(min(10, len(YT_comments)), random_state=SEED)
)

## 6. Exploratory Text Analysis

Bagian ini digunakan untuk melihat:
- panjang komentar
- volume komentar per tanggal
- kata yang paling sering muncul
- distribusi sentimen jika label sudah tersedia

Tidak ada insight tekstual yang ditulis sebelum grafik/tabel dihitung dari dataset.

In [ ]:
YT_comments["comment_length"] = YT_comments["text_preprocessed"].str.split().str.len()

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(YT_comments["comment_length"].clip(upper=100), bins=30)
ax.set_title("Distribusi Panjang Komentar")
ax.set_xlabel("Jumlah token (maksimum ditampilkan 100)")
ax.set_ylabel("Jumlah komentar")
plt.tight_layout()
plt.show()

daily_volume = (
    YT_comments.dropna(subset=["published_at"])
    .assign(date=lambda d: d["published_at"].dt.date)
    .groupby("date")
    .size()
)

fig, ax = plt.subplots(figsize=(12, 5))
daily_volume.plot(ax=ax)
ax.set_title("Volume Komentar YouTube dari Waktu ke Waktu")
ax.set_xlabel("Tanggal")
ax.set_ylabel("Jumlah komentar")
plt.tight_layout()
plt.show()

In [ ]:
all_words = " ".join(YT_comments["text_preprocessed"].dropna())
word_counts = Counter(all_words.split())

top_words_overall = pd.DataFrame(
    word_counts.most_common(20),
    columns=["word", "count"]
)

display(top_words_overall)

In [ ]:
plt.figure(figsize=(10, 7))
sns.barplot(
    data=top_words_overall.sort_values("count"),
    x="count",
    y="word"
)
plt.title("20 Kata Paling Sering Muncul")
plt.xlabel("Frekuensi")
plt.ylabel("Kata")
plt.tight_layout()
plt.show()

## 7. Sentiment Distribution

Bagian ini hanya dijalankan ketika `sentiment` sudah berisi label valid.

**Important:** notebook tidak lagi menggunakan Indonesian Sentiment Lexicon/InSet sebagai ground-truth label. Label supervised harus berasal dari proses labeling dataset.

In [ ]:
if not LABELS_READY:
    print("Belum ada sentiment label. Distribusi sentimen belum dapat dihitung.")
else:
    sentiment_counts = (
        YT_labeled["sentiment"]
        .value_counts()
        .reindex(["positive", "negative", "neutral"])
        .dropna()
    )

    sentiment_pct = (sentiment_counts / sentiment_counts.sum() * 100).round(2)

    sentiment_summary = pd.DataFrame({
        "count": sentiment_counts.astype(int),
        "percentage": sentiment_pct
    })

    display(sentiment_summary)

    plt.figure(figsize=(7, 5))
    sns.barplot(
        x=sentiment_summary.index,
        y=sentiment_summary["count"]
    )
    plt.title("Distribusi Label Sentimen")
    plt.xlabel("Sentiment")
    plt.ylabel("Jumlah komentar")
    plt.tight_layout()
    plt.show()

## 8. Sentiment Trend

Untuk komentar yang sudah memiliki label, kita dapat melihat perubahan volume **positive / negative / neutral** berdasarkan `published_at`.

Trend menunjukkan **kapan** diskusi berubah; penyebab lonjakan harus dianalisis terpisah dengan sumber eksternal dan tidak boleh disimpulkan hanya karena dua kejadian terjadi pada tanggal yang berdekatan.

In [ ]:
if not LABELS_READY:
    print("Belum ada label sentiment. Gunakan daily_volume untuk melihat volume diskusi.")
else:
    sentiment_trend = (
        YT_labeled.dropna(subset=["published_at"])
        .assign(date=lambda d: d["published_at"].dt.date)
        .groupby(["date", "sentiment"])
        .size()
        .unstack(fill_value=0)
    )

    for label in ["positive", "negative", "neutral"]:
        if label not in sentiment_trend.columns:
            sentiment_trend[label] = 0

    sentiment_trend = sentiment_trend[["positive", "negative", "neutral"]]

    fig, ax = plt.subplots(figsize=(13, 5))
    sentiment_trend.plot(ax=ax)
    ax.set_title("Tren Sentimen Komentar YouTube")
    ax.set_xlabel("Tanggal")
    ax.set_ylabel("Jumlah komentar")
    plt.tight_layout()
    plt.show()

## 9. Prepare Binary Classification Dataset

Model deep learning mengikuti pendekatan project template: klasifikasi dua kelas.

Encoding:
- `positive = 0`
- `negative = 1`

Komentar `neutral` tidak digunakan dalam binary model, tetapi tetap ada pada `YT_comments`.

In [ ]:
BINARY_LABEL_MAP = {"positive": 0, "negative": 1}

if BINARY_LABELS_READY:
    YT_comments_binary = (
        YT_labeled[
            YT_labeled["sentiment"].isin(BINARY_LABEL_MAP.keys())
            & (YT_labeled["text_preprocessed"].str.len() > 0)
        ]
        .copy()
    )

    YT_comments_binary["label"] = YT_comments_binary["sentiment"].map(BINARY_LABEL_MAP)

    texts = YT_comments_binary["text_preprocessed"].values
    y = YT_comments_binary["label"].values.astype(int)

    print("Binary dataset:", YT_comments_binary.shape)
    print("\nBinary label distribution:")
    print(YT_comments_binary["sentiment"].value_counts())
else:
    YT_comments_binary = pd.DataFrame()
    texts = np.array([])
    y = np.array([], dtype=int)
    print("Binary modeling belum siap. Isi label positive/negative terlebih dahulu.")

## 10. Train-Test Split

Tokenizer hanya dipelajari dari **training set** untuk mengurangi risiko data leakage.

In [ ]:
if BINARY_LABELS_READY:
    from sklearn.model_selection import train_test_split

    X_train, X_test, y_train, y_test = train_test_split(
        texts,
        y,
        test_size=0.20,
        random_state=SEED,
        stratify=y
    )

    print("X_train:", X_train.shape)
    print("X_test :", X_test.shape)
    print("y_train:", y_train.shape)
    print("y_test :", y_test.shape)
else:
    X_train = X_test = np.array([])
    y_train = y_test = np.array([], dtype=int)

## 11. Tokenization & Padding

Model menggunakan:
**Tokenizer → Sequence → Padding → Embedding**

`MAX_LENGTH = 200` digunakan sebagai batas panjang sequence agar input model memiliki ukuran yang konsisten.

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_VOCAB_SIZE = 20000
MAX_LENGTH = 200

if BINARY_LABELS_READY:
    tokenizer = Tokenizer(
        num_words=MAX_VOCAB_SIZE,
        oov_token="<OOV>"
    )
    tokenizer.fit_on_texts(X_train)

    train_sequences = tokenizer.texts_to_sequences(X_train)
    test_sequences = tokenizer.texts_to_sequences(X_test)

    X_train_padded = pad_sequences(
        train_sequences,
        maxlen=MAX_LENGTH,
        padding="post",
        truncating="post"
    )
    X_test_padded = pad_sequences(
        test_sequences,
        maxlen=MAX_LENGTH,
        padding="post",
        truncating="post"
    )

    vocab_size = min(
        MAX_VOCAB_SIZE,
        len(tokenizer.word_index) + 1
    )

    print("Vocabulary size:", vocab_size)
    print("Padded train shape:", X_train_padded.shape)
    print("Padded test shape :", X_test_padded.shape)
else:
    tokenizer = None
    X_train_padded = X_test_padded = np.empty((0, MAX_LENGTH), dtype=np.int32)
    vocab_size = 0

## 12. Baseline Deep Learning Model — Bidirectional LSTM

Arsitektur:
- Embedding
- Bidirectional LSTM
- Dropout
- Dense
- Sigmoid output

Model menghasilkan probabilitas antara 0 dan 1 untuk klasifikasi binary.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

tf.random.set_seed(SEED)

if BINARY_LABELS_READY:
    classes = np.unique(y_train)
    class_weights_values = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y_train
    )
    class_weight = dict(zip(classes, class_weights_values))

    baseline_model = Sequential([
        Embedding(
            input_dim=vocab_size,
            output_dim=128,
            input_length=MAX_LENGTH
        ),
        Bidirectional(LSTM(64, return_sequences=True)),
        Dropout(0.5),
        Bidirectional(LSTM(32)),
        Dropout(0.3),
        Dense(32, activation="relu"),
        Dense(1, activation="sigmoid")
    ])

    baseline_model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    baseline_checkpoint = ModelCheckpoint(
        "best_baseline_bilstm.keras",
        save_best_only=True,
        monitor="val_accuracy",
        mode="max"
    )

    baseline_early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    )

    baseline_history = baseline_model.fit(
        X_train_padded,
        y_train,
        epochs=20,
        batch_size=64,
        validation_split=0.20,
        class_weight=class_weight,
        callbacks=[baseline_early_stopping, baseline_checkpoint],
        verbose=1
    )

    baseline_prob = baseline_model.predict(
        X_test_padded, verbose=0
    ).ravel()
    baseline_pred = (baseline_prob >= 0.5).astype(int)

    print(classification_report(
        y_test,
        baseline_pred,
        target_names=["positive", "negative"]
    ))
else:
    baseline_model = None
    baseline_history = None
    print("Baseline model belum dijalankan karena label binary belum tersedia.")

In [ ]:
if BINARY_LABELS_READY and baseline_model is not None:
    baseline_f1_macro = f1_score(y_test, baseline_pred, average="macro")
    baseline_f1_negative = f1_score(y_test, baseline_pred, pos_label=1)

    baseline_accuracy = accuracy_score(y_test, baseline_pred)

    print(f"Baseline Accuracy : {baseline_accuracy:.4f}")
    print(f"Baseline Macro F1 : {baseline_f1_macro:.4f}")
    print(f"Negative-class F1 : {baseline_f1_negative:.4f}")

    cm = confusion_matrix(y_test, baseline_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=["positive", "negative"],
        yticklabels=["positive", "negative"]
    )
    plt.title("Confusion Matrix — Baseline BiLSTM")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()

## 13. Hyperparameter Tuning with Optuna

Optuna digunakan untuk mencari kombinasi hyperparameter pada model **Bidirectional GRU**.

Hyperparameter yang dituning:
- embedding dimension
- GRU units
- dropout
- learning rate
- L2 regularization

Perbaikan dibanding notebook template:
- objective menggunakan padded training data yang benar
- `suggest_float(..., log=True)` digunakan untuk parameter log-scale
- learning rate terbaik benar-benar dipakai saat compile final model
- data test tidak digunakan selama tuning

In [ ]:
from tensorflow.keras.regularizers import l2

RUN_OPTUNA = False
N_TRIALS = 10
TUNE_EPOCHS = 8

def build_bigru_model(params):
    model = Sequential([
        Embedding(
            input_dim=vocab_size,
            output_dim=params["embedding_dim"],
            input_length=MAX_LENGTH
        ),
        Bidirectional(
            GRU(
                params["gru_units1"],
                return_sequences=True,
                kernel_regularizer=l2(params["l2_reg"])
            )
        ),
        Dropout(params["dropout_rate1"]),
        Bidirectional(
            GRU(
                params["gru_units2"],
                kernel_regularizer=l2(params["l2_reg"])
            )
        ),
        Dropout(params["dropout_rate2"]),
        Dense(
            32,
            activation="relu",
            kernel_regularizer=l2(params["l2_reg"])
        ),
        Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=Adam(learning_rate=params["learning_rate"]),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

if BINARY_LABELS_READY and RUN_OPTUNA:
    import optuna

    def objective(trial):
        tf.keras.backend.clear_session()

        params = {
            "embedding_dim": trial.suggest_categorical(
                "embedding_dim", [64, 128, 256]
            ),
            "gru_units1": trial.suggest_categorical(
                "gru_units1", [32, 64, 128]
            ),
            "gru_units2": trial.suggest_categorical(
                "gru_units2", [16, 32, 64]
            ),
            "dropout_rate1": trial.suggest_float(
                "dropout_rate1", 0.2, 0.6
            ),
            "dropout_rate2": trial.suggest_float(
                "dropout_rate2", 0.2, 0.6
            ),
            "learning_rate": trial.suggest_float(
                "learning_rate", 1e-4, 1e-2, log=True
            ),
            "l2_reg": trial.suggest_float(
                "l2_reg", 1e-5, 1e-2, log=True
            )
        }

        model = build_bigru_model(params)

        history = model.fit(
            X_train_padded,
            y_train,
            epochs=TUNE_EPOCHS,
            batch_size=64,
            validation_split=0.20,
            class_weight=class_weight,
            callbacks=[
                EarlyStopping(
                    monitor="val_loss",
                    patience=2,
                    restore_best_weights=True
                )
            ],
            verbose=0
        )

        return max(history.history["val_accuracy"])

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED)
    )
    study.optimize(objective, n_trials=N_TRIALS)

    best_params = study.best_params
    print("Best validation accuracy:", study.best_value)
    print("Best hyperparameters:")
    for key, value in best_params.items():
        print(f"  {key}: {value}")
else:
    study = None
    best_params = None
    print(
        "Optuna belum dijalankan. Set RUN_OPTUNA = True "
        "untuk mengaktifkan tuning."
    )

## 14. Final Bidirectional GRU

Model final menggunakan hyperparameter hasil Optuna bila tuning dijalankan.

Untuk menjaga reproducibility, model final tidak menggunakan angka hard-coded dari project lain.

In [ ]:
if BINARY_LABELS_READY:
    if best_params is None:
        best_params = {
            "embedding_dim": 128,
            "gru_units1": 64,
            "gru_units2": 32,
            "dropout_rate1": 0.4,
            "dropout_rate2": 0.4,
            "learning_rate": 0.001,
            "l2_reg": 0.0001
        }
        print("Menggunakan default parameters karena Optuna belum dijalankan.")

    final_model = build_bigru_model(best_params)

    final_checkpoint = ModelCheckpoint(
        "best_model.keras",
        save_best_only=True,
        monitor="val_accuracy",
        mode="max"
    )

    final_early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

    final_history = final_model.fit(
        X_train_padded,
        y_train,
        epochs=20,
        batch_size=64,
        validation_split=0.20,
        class_weight=class_weight,
        callbacks=[final_early_stopping, final_checkpoint],
        verbose=1
    )

    test_loss, test_accuracy = final_model.evaluate(
        X_test_padded,
        y_test,
        verbose=0
    )

    test_prob = final_model.predict(
        X_test_padded, verbose=0
    ).ravel()
    test_pred = (test_prob >= 0.5).astype(int)

    final_f1_macro = f1_score(
        y_test,
        test_pred,
        average="macro"
    )

    print(f"Test loss     : {test_loss:.4f}")
    print(f"Test accuracy : {test_accuracy:.4f}")
    print(f"Test Macro F1 : {final_f1_macro:.4f}")
    print("\nClassification Report:")
    print(classification_report(
        y_test,
        test_pred,
        target_names=["positive", "negative"],
        zero_division=0
    ))
else:
    final_model = None
    final_history = None
    final_f1_macro = np.nan
    print("Final model belum dijalankan karena label binary belum tersedia.")

In [ ]:
if BINARY_LABELS_READY and final_model is not None:
    cm = confusion_matrix(y_test, test_pred)

    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=["positive", "negative"],
        yticklabels=["positive", "negative"]
    )
    plt.title("Confusion Matrix — Final BiGRU")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()

In [ ]:
if BINARY_LABELS_READY and final_history is not None:
    history_df = pd.DataFrame(final_history.history)

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(history_df["accuracy"], label="train_accuracy")
    ax.plot(history_df["val_accuracy"], label="val_accuracy")
    ax.set_title("Training vs Validation Accuracy")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy")
    ax.legend()
    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(history_df["loss"], label="train_loss")
    ax.plot(history_df["val_loss"], label="val_loss")
    ax.set_title("Training vs Validation Loss")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 15. Model Comparison

Perbandingan dilakukan dari hasil yang benar-benar dihitung pada dataset ini.

Jangan menyalin angka accuracy/F1 dari notebook lain.

In [ ]:
if BINARY_LABELS_READY and final_model is not None:
    comparison = pd.DataFrame({
        "model": ["BiLSTM baseline", "BiGRU final"],
        "accuracy": [
            accuracy_score(y_test, baseline_pred),
            accuracy_score(y_test, test_pred)
        ],
        "macro_f1": [
            baseline_f1_macro,
            final_f1_macro
        ]
    })

    display(
        comparison.sort_values("macro_f1", ascending=False)
        .reset_index(drop=True)
    )
else:
    print("Model comparison belum tersedia karena label binary belum siap.")

## 16. Top Comments by Sentiment

Tabel ini membantu membaca contoh komentar yang masuk ke masing-masing kelas. Contoh komentar harus dipahami sebagai **sampel dari dataset**, bukan sebagai representasi seluruh masyarakat.

In [ ]:
if LABELS_READY:
    for label in ["positive", "negative", "neutral"]:
        subset = YT_labeled[YT_labeled["sentiment"] == label]
        if len(subset) == 0:
            continue

        print(f"\n=== {label.upper()} ===")
        display(
            subset[
                ["comment", "video_title", "channel_title", "published_at"]
            ].head(10)
        )
else:
    print("Contoh berdasarkan sentimen belum tersedia karena label belum ada.")

## 17. Interpretation Guidelines

Saat menulis hasil akhir:

- **Data:** laporkan angka atau observasi yang langsung terlihat dari dataset.
- **Analysis:** jelaskan perbandingan, proporsi, atau pola yang dihitung.
- **Insight:** jelaskan mengapa pola tersebut penting dalam konteks analisis.
- **Hypothesis:** tandai dugaan penyebab yang membutuhkan verifikasi lebih lanjut.
- **Recommendation:** hanya dibuat bila ada evidence yang cukup.

Hindari menyimpulkan penyebab suatu lonjakan hanya dari korelasi tanggal.

## 18. Limitations

1. Dataset berasal dari komentar YouTube yang berhasil dikumpulkan, sehingga tidak mewakili seluruh populasi.
2. Komentar online dapat mengandung slang, sarcasm, spam, atau konteks yang sulit dipahami model.
3. Label supervised pada dataset ini berasal dari **AI-assisted semantic labeling**, sehingga kualitas model bergantung pada kualitas dan konsistensi proses labeling tersebut.
4. Distribusi kelas tidak seimbang; terutama kelas positive yang jauh lebih sedikit daripada negative dan neutral. Notebook menggunakan class weight untuk membantu training binary model, tetapi metrik tetap perlu dibaca dengan hati-hati.
5. Binary classification mengeluarkan neutral dari training, sehingga model hanya belajar membedakan positive vs negative.
6. Model RNN berbasis token sequence memiliki keterbatasan dalam memahami konteks yang sangat panjang, sarcasm, dan bahasa yang sangat tidak baku.

## 19. Final Output

Setelah seluruh cell selesai dijalankan, project menghasilkan:

- cleaned/preprocessed text
- distribusi sentimen
- trend diskusi
- baseline BiLSTM
- final Bidirectional GRU
- classification report
- confusion matrix
- model comparison
- contoh komentar per sentimen

Semua angka, grafik, dan insight pada laporan akhir harus berasal dari eksekusi notebook ini terhadap:

`data/mbg_comments_labeled.csv`.

## 20. Reproducibility Checklist

Sebelum memasukkan hasil ke portfolio:

- [ ] `mbg_comments_labeled.csv` tersedia di `data/`.
- [ ] Kolom `sentiment` hanya menggunakan label yang valid.
- [ ] Labeling method dijelaskan sebagai AI-assisted semantic labeling.
- [ ] Tidak ada hasil lama dari project lain yang tertinggal.
- [ ] Notebook dijalankan dari awal sampai akhir tanpa error.
- [ ] Test set tidak digunakan untuk tuning.
- [ ] Accuracy dan F1 dihitung dari test set.
- [ ] Macro F1 digunakan untuk perbandingan model karena distribusi kelas binary tidak seimbang.
- [ ] Insight ditulis berdasarkan output aktual.
- [ ] Tidak ada angka hasil copy dari notebook template.